# 🤖 Notebook 3 — Training Model & Evaluasi
**Kelompok 1 — Tugas Besar Analisis Big Data**

Tahapan:
1. Training Random Forest Classifier (PySpark MLlib)
2. Evaluasi: Accuracy, F1-Score, Precision, Recall
3. Confusion Matrix
4. Feature Importance

In [ ]:
from pyspark.sql import SparkSession
from pyspark.ml.classification import RandomForestClassifier
from pyspark.ml.evaluation import MulticlassClassificationEvaluator, BinaryClassificationEvaluator
import matplotlib.pyplot as plt
import matplotlib
import numpy as np

matplotlib.rcParams['figure.dpi'] = 120

spark = SparkSession.builder \
    .appName('WaterPotability_Modeling') \
    .master('local[*]') \
    .config('spark.driver.memory', '2g') \
    .getOrCreate()

print('SparkSession berhasil dibuat.')

---
## 3.1 Membaca Dataset yang Sudah Diproses

In [ ]:
train_df = spark.read.parquet('/home/jovyan/work/output/train_data.parquet')
test_df  = spark.read.parquet('/home/jovyan/work/output/test_data.parquet')

print(f'Training : {train_df.count():,} sampel')
print(f'Testing  : {test_df.count():,} sampel')
train_df.show(3, truncate=True)

---
## 3.2 Training Random Forest Classifier

In [ ]:
rf = RandomForestClassifier(
    labelCol='Potability',
    featuresCol='features',
    numTrees=100,          # jumlah pohon keputusan
    maxDepth=10,           # kedalaman maksimum tiap pohon
    minInstancesPerNode=2, # minimum sampel per daun
    featureSubsetStrategy='sqrt',  # strategi pemilihan fitur
    seed=42
)

print('Training Random Forest...')
print(f'  numTrees = {rf.getNumTrees}')
print(f'  maxDepth = {rf.getMaxDepth()}')

rf_model = rf.fit(train_df)
print('\n✅ Model berhasil dilatih!')

---
## 3.3 Prediksi pada Data Testing

In [ ]:
predictions = rf_model.transform(test_df)

print('=== SAMPLE PREDIKSI ===')
predictions.select('Potability', 'prediction', 'probability').show(10, truncate=False)

---
## 3.4 Evaluasi Metrik

In [ ]:
evaluator = MulticlassClassificationEvaluator(
    labelCol='Potability',
    predictionCol='prediction'
)

accuracy  = evaluator.evaluate(predictions, {evaluator.metricName: 'accuracy'})
f1        = evaluator.evaluate(predictions, {evaluator.metricName: 'f1'})
precision = evaluator.evaluate(predictions, {evaluator.metricName: 'weightedPrecision'})
recall    = evaluator.evaluate(predictions, {evaluator.metricName: 'weightedRecall'})

# AUC-ROC
bin_evaluator = BinaryClassificationEvaluator(
    labelCol='Potability',
    rawPredictionCol='rawPrediction',
    metricName='areaUnderROC'
)
auc_roc = bin_evaluator.evaluate(predictions)

print('=' * 50)
print('       HASIL EVALUASI MODEL RANDOM FOREST')
print('=' * 50)
print(f'  Accuracy          : {accuracy:.4f}  ({accuracy*100:.2f}%)')
print(f'  F1-Score          : {f1:.4f}  ({f1*100:.2f}%)')
print(f'  Precision         : {precision:.4f}  ({precision*100:.2f}%)')
print(f'  Recall            : {recall:.4f}  ({recall*100:.2f}%)')
print(f'  AUC-ROC           : {auc_roc:.4f}  ({auc_roc*100:.2f}%)')
print('=' * 50)

# Perbandingan dengan target
print('\n=== PERBANDINGAN DENGAN TARGET ===')
targets = {'Accuracy': 0.95, 'F1-Score': 0.94, 'Precision': 0.94, 'Recall': 0.94}
results = {'Accuracy': accuracy, 'F1-Score': f1, 'Precision': precision, 'Recall': recall}
for metric, target in targets.items():
    val = results[metric]
    status = '✅ TERCAPAI' if val >= target else '❌ BELUM TERCAPAI'
    print(f'  {metric:<12}: {val:.4f} (target ≥ {target}) {status}')

---
## 3.5 Confusion Matrix

In [ ]:
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay

y_true = [int(row.Potability) for row in predictions.select('Potability').collect()]
y_pred = [int(row.prediction) for row in predictions.select('prediction').collect()]

cm = confusion_matrix(y_true, y_pred)

fig, ax = plt.subplots(figsize=(7, 5))
disp = ConfusionMatrixDisplay(
    confusion_matrix=cm,
    display_labels=['Tidak Layak (0)', 'Layak (1)']
)
disp.plot(
    cmap='Blues',
    ax=ax,
    colorbar=True,
    values_format='d'
)
ax.set_title('Confusion Matrix — Random Forest Classifier', fontsize=13, fontweight='bold', pad=15)
plt.tight_layout()
plt.savefig('/home/jovyan/work/output/figures/confusion_matrix.png', bbox_inches='tight', dpi=150)
plt.show()

# Detail confusion matrix
tn, fp, fn, tp = cm.ravel()
print(f'\nTrue Negative  (TN) : {tn:,}  — Tidak layak, diprediksi benar')
print(f'False Positive (FP) : {fp:,}  — Sebenarnya tidak layak, diprediksi layak')
print(f'False Negative (FN) : {fn:,}  — Sebenarnya layak, diprediksi tidak layak')
print(f'True Positive  (TP) : {tp:,}  — Layak, diprediksi benar')
print('✅ Gambar disimpan: output/figures/confusion_matrix.png')

---
## 3.6 Feature Importance

In [ ]:
FEATURE_COLS = [
    'ph', 'Hardness', 'Solids', 'Chloramines', 'Sulfate',
    'Conductivity', 'Organic_carbon', 'Trihalomethanes', 'Turbidity'
]

importances = rf_model.featureImportances.toArray()
feat_imp_df = sorted(zip(FEATURE_COLS, importances), key=lambda x: x[1], reverse=True)

print('=== FEATURE IMPORTANCE (diurutkan) ===')
for rank, (feat, imp) in enumerate(feat_imp_df, 1):
    bar = '█' * int(imp * 200)
    print(f'  {rank}. {feat:<20} {imp:.4f}  {bar}')

# Plot
features_sorted = [x[0] for x in feat_imp_df]
values_sorted   = [x[1] for x in feat_imp_df]

colors = plt.cm.RdYlGn(np.linspace(0.3, 0.9, len(features_sorted))[::-1])

fig, ax = plt.subplots(figsize=(10, 6))
bars = ax.barh(features_sorted[::-1], values_sorted[::-1], color=colors[::-1], edgecolor='black', linewidth=0.5)

for bar, val in zip(bars, values_sorted[::-1]):
    ax.text(val + 0.002, bar.get_y() + bar.get_height()/2,
            f'{val:.4f}', va='center', fontsize=9)

ax.set_xlabel('Feature Importance Score', fontsize=11)
ax.set_title('Feature Importance — Random Forest Classifier', fontsize=13, fontweight='bold')
ax.set_xlim(0, max(values_sorted) * 1.2)
ax.grid(axis='x', alpha=0.3)
plt.tight_layout()
plt.savefig('/home/jovyan/work/output/figures/feature_importance.png', bbox_inches='tight', dpi=150)
plt.show()
print('✅ Gambar disimpan: output/figures/feature_importance.png')

---
## 3.7 Simpan Model (Opsional)

In [ ]:
MODEL_PATH = '/home/jovyan/work/output/model/random_forest_model'
rf_model.write().overwrite().save(MODEL_PATH)
print(f'✅ Model disimpan ke: {MODEL_PATH}')
print('\n➡️ Lanjut ke Notebook 04: Visualisasi & Perbandingan Baseline')

spark.stop()